# 05 — k-NN: të nxënit nga fqinjët

## Konteksti dhe problemi shkencor

Regjimet e një sistemi të zbehur përdoren për të eksploruar distancën, shkallëzimin, parametrin k dhe dimensionin.

### Objektivat e të nxënit

- të implementoni k-NN;
- të kuptoni rolin e distancës dhe scaling;
- të shihni bias–variance përmes k;
- të demonstroni mallkimin e dimensionalitetit.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Regjime dinamike si problem lokal

In [ ]:
df=pd.read_csv(DATA/'dynamical_regimes.csv'); X=df[['frekuenca_Hz','zbehja']].to_numpy(); y=df.regjimi.to_numpy(); plt.scatter(X[:,0],X[:,1],c=y,s=16); plt.show()

## 2. Distanca dhe implementimi manual

In [ ]:
from src.neighbors.manual import knn_predict
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=0); sc=StandardScaler().fit(Xtr); Xtrs=sc.transform(Xtr); Xtes=sc.transform(Xte); pred=knn_predict(Xtrs,ytr,Xtes,k=5); print(np.mean(pred==yte))

## 3. Krahasim me bibliotekën

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
sk=KNeighborsClassifier(5).fit(Xtrs,ytr); print(np.mean(sk.predict(Xtes)==pred),sk.score(Xtes,yte))

## 4. Zgjedhja e k me cross-validation

In [ ]:
ks=range(1,41,2); means=[]; stds=[]
for k in ks:
 p=make_pipeline(StandardScaler(),KNeighborsClassifier(k)); s=cross_val_score(p,X,y,cv=5); means.append(s.mean()); stds.append(s.std())
plt.errorbar(list(ks),means,yerr=stds,marker='o'); plt.xlabel('k'); plt.ylabel('CV accuracy'); plt.show()

## 5. Efekti i shkallëzimit

In [ ]:
X_bad=X.copy(); X_bad[:,0]*=1000; print('pa scaling',cross_val_score(KNeighborsClassifier(7),X_bad,y,cv=5).mean()); print('me scaling',cross_val_score(make_pipeline(StandardScaler(),KNeighborsClassifier(7)),X_bad,y,cv=5).mean())

## 6. Mallkimi i dimensionalitetit

In [ ]:
rng=np.random.default_rng(1); vals=[]
for q in [0,2,5,10,30,80]:
 Xq=np.c_[X,rng.normal(size=(len(X),q))] if q else X
 vals.append((q,cross_val_score(make_pipeline(StandardScaler(),KNeighborsClassifier(7)),Xq,y,cv=5).mean()))
pd.DataFrame(vals,columns=['noise_features','cv_accuracy'])

## 7. Kufiri i vendimit

In [ ]:
pipe=make_pipeline(StandardScaler(),KNeighborsClassifier(7)).fit(X,y); xx,yy=np.meshgrid(np.linspace(X[:,0].min()-.2,X[:,0].max()+.2,250),np.linspace(X[:,1].min()-.08,X[:,1].max()+.08,250)); zz=pipe.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape); plt.contourf(xx,yy,zz,alpha=.25); plt.scatter(X[:,0],X[:,1],c=y,s=12); plt.show()

## 8. Kostoja e parashikimit

In [ ]:
for n in [100,1000,5000]:
 rng=np.random.default_rng(2); Xt=rng.normal(size=(n,15)); yt=rng.integers(0,2,n); z=rng.normal(size=(200,15)); m=KNeighborsClassifier(5).fit(Xt,yt); t=time.perf_counter(); m.predict(z); print(n,time.perf_counter()-t)

## 9. Ushtrime

- Implementoni votim të ponderuar me $1/(d+\epsilon)$.
- Krahasoni Manhattan dhe Euclidean.
- Shtoni veçori zhurme derisa accuracy të bjerë nën 0.8.
- Gjeni shembuj testimi dhe listoni fqinjët që prodhuan vendimin.

## 10. Shembull i plotë

Zgjidhni k me CV vetëm në development set, pastaj raportoni testin final dhe shpjegoni pse scaling është pjesë e pipeline.

## 11. Playground

Krijoni një regjim të katërt që mbivendoset fort me të dytin. A ndihmon k më i madh apo më i vogël?